# Evaluate Sample Path Forecasts of the Car Parts Dataset

This notebook shows how SPEV can be used to extend a FEV task to evaluate sample path forecasts instead of marginal quantile forecasts. The overall approach is the following: First, define a FEV task as usual. Here, use the Car Parts data from Chronos datasets as a small example. The same data was used to evaluate sample paths in, among others, [DeepAR: Probabilistic Forecasting with Autoregressive Recurrent Networks](https://arxiv.org/abs/1704.04110) (Salinas et al., 2017). Second, define a SPEV task using the FEV task as a basis. The SPEV task will use the same seasonality, windows, and horizon, and data. At that point, you are ready to evaluate forecasts using `SamplePathTask.evaluation_summary()`. This notebook shows how quantile predictions from a Chronos model can be adapted to sample paths for evaluation.

In [ ]:
import numpy as np

from tqdm import tqdm
from fev import Task
from spev import SamplePathTask

# Import additional helpers needed for this example but not in general.
# Ensure to install the required dependencies using `spev[adapter,models]`.
from spev.adapter import reshape_fev_predictions_to_sample_paths
from spev.models import Chronos2Model

## Define the FEV Task

The FEV task is the basis for the evaluation. The SPEV sample path task only extends it to evaluate the same task but with sample path evaluation metrics.

In [ ]:
base_task = Task(
    dataset_path="autogluon/chronos_datasets",
    dataset_config="monash_car_parts",
    horizon=8,
    num_windows=1,
    seasonality=12,
    quantile_levels=[
        0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5,
        0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99
    ],
    target="target",
)

## Define the SPEV Task

The sample path task extends the base task by defining aggregates of the sample paths and metrics of the aggregates. The `SUM` aggregates and quantile loss metrics implement the evaluation applied onto the car parts dataset in the aforementioned Salinas et al. (2017). The additional aggregates `ANY_GT` and `COUNT_BETWEEN` highlight additional options.

In [ ]:
# Initialization of the `SamplePathTask` will download the dataset
# defined in the `Task` (in this case from HuggingFace Hub).

task = SamplePathTask(
    base=base_task,
    aggregates=[
        {"name": "SUM", "lead_time": 0, "span": 1},
        {"name": "SUM", "lead_time": 2, "span": 1},
        {"name": "SUM", "lead_time": 0, "span": 8},
        {"name": "ANY_GT", "lead_time": 0, "span": 8, "k": 4},
        {"name": "COUNT_BETWEEN", "lead_time": 2, "span": 4, "lb": 1, "ub": 3},
    ],
    metrics=[
        {"name": "WCRPS", "epsilon": 0.01},
        {"name": "WQL", "quantile_level": 0.9, "epsilon": 0.01},
        {"name": "WQL", "quantile_level": 0.5, "epsilon": 0.01},
    ]
)

## Forecast the FEV Task Using Chronos-2

Derive marginal quantile forecasts using a Chronos-2 model. Since the SPEV task only adds aggregates and metrics, the predicition step depends on the information of the FEV task, only. 

In [ ]:
model = Chronos2Model(device="cpu") # No need for a GPU in this example
chronos_predictions_for_fev = model.fit_predict(base_task)

print(chronos_predictions_for_fev)

## Turn Chronos-2 Quantile Forecasts into Sample Paths

However, to evaluate the Chronos-2 predictions on the SPEV task, they need to be turned into sample paths. This can be done using [Shuffler](https://github.com/timradtke/shuffler) which is an optional dependency for this purpose of adapting quantiles into sample paths. The evaluation becomes one of the combination of Chronos-2 and Shuffler.

In [ ]:
sample_paths_per_window = reshape_fev_predictions_to_sample_paths(
    predictions_per_window=chronos_predictions_for_fev,
    task=base_task,
    n_samples=2501,
    force_marginal=False,
)

The `reshape_fev_predictions_to_sample_paths()` function is designed to return sample paths in the shape required by `SamplePathTask.evaluation_summary()`. The `sample_paths_per_window` object is a list with one array per forecast window defined in the FEV task (in this case, one). Each array is four-dimensional: The first dimension is the number of time series, the second dimension is the number of samples, the third dimension is the length of the forecast horizon, and the fourth is the number of targets. In this case, the shape will be (2674, 2501, 8, 1).

## Evaluate Using SPEV Task

Pass the sample path forecasts to the SPEV task for evaluation. It returns a list of evaluated metrics, one for each combination of aggregates and metrics defined in the sample path task. Here, with 5 aggregates and 3 metrics, there will be 15 evaluated metrics.

In [ ]:
task.evaluation_summary(
    y_pred_per_window=sample_paths_per_window,
)

Let's compare those results against a constant-zero forecast. As described above, the forecasts are a list of four-dimensional arrays, which are easy to create:

In [ ]:
task.evaluation_summary(
    y_pred_per_window=[np.zeros(shape=(2674, 2501, 8, 1))],
)